# NYC Housing x Transportation: Data Story (Milestone 1)
**Course:** Data Engineering and Visualization, Winter 2026 | **Deadline:** 9 Nov 2026

**How to work in this notebook (read first)**
- Each member works ONLY inside their own section (marked with their name) to avoid edit clashes.
- Do not rename the shared variables: `df_housing_raw`, `df_transport_raw`, `df_housing`, `df_transport`, `merged`.
- Every decision (drop vs impute, outlier rule, join level) needs a markdown cell explaining **why**.
- Log every AI prompt you use in section 9.

| Member | Section |
|---|---|
| M1 Housing lead | 2, 3, 7 (RQ1-2) |
| M2 Transport lead | 2, 4, 7 (RQ3-4) |
| M3 Integration lead | 0, 1, 5, 6, 7 (RQ5-6) |
| M4 Website + story lead | 7 (RQ7-8), 8, 10 |

## 0. Setup and dataset links (M3)
- Housing: https://data.cityofnewyork.us/Housing-Development/DOB-NOW-Build-Job-Application-Filings/w9ak-ipjd
- Crashes: https://data.cityofnewyork.us/Public-Safety/Motor-Vehicle-Collisions-Crashes/h9gi-nx95

Run the next two cells first. The download takes a few minutes; everyone should run it once in their own Colab copy.

In [ ]:
import re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from urllib.parse import urlencode

# Both IDs verified on data.cityofnewyork.us (Oct 2026):
HOUSING_ID   = 'w9ak-ipjd'   # DOB NOW: Build - Job Application Filings (~969K rows, 95 columns)
TRANSPORT_ID = 'h9gi-nx95'   # Motor Vehicle Collisions - Crashes (~2.27M rows, 29 columns)
START_DATE   = '2022-01-01'  # sampling window start (justified in section 1)
API = 'https://data.cityofnewyork.us/resource/{}.csv'

def find_col(cols, candidates, regex=None, required=True):
    """Return the first matching column name. Column names differ between the CSV
    download and the API, so we look them up instead of hard-coding them."""
    cols = list(cols)
    for c in candidates:
        if c in cols:
            return c
    if regex:
        hits = [c for c in cols if re.search(regex, c, flags=re.I)]
        if hits:
            return hits[0]
    if required:
        raise KeyError(f'None of {candidates} (regex={regex}) found. Available columns: {cols}')
    return None

def soda_get(dataset_id, params):
    return pd.read_csv(API.format(dataset_id) + '?' + urlencode(params), low_memory=False)

def probe_columns(dataset_id):
    """Download 5 rows just to learn the real column names."""
    return list(soda_get(dataset_id, {'$limit': 5}).columns)

def load_since(dataset_id, date_col, start=START_DATE, page=100_000, max_rows=2_000_000):
    """Download rows with date_col >= start in pages of 100k (stable order by :id).
    Paging avoids timeouts and Colab memory spikes; the full tables are too big to pull at once."""
    pages, offset = [], 0
    while offset < max_rows:
        chunk = soda_get(dataset_id, {'$where': f"{date_col} >= '{start}T00:00:00'",
                                      '$order': ':id', '$limit': page, '$offset': offset})
        pages.append(chunk)
        print(f'  {dataset_id}: fetched {offset + len(chunk):,} rows')
        if len(chunk) < page:
            break
        offset += page
    return pd.concat(pages, ignore_index=True)

In [ ]:
housing_cols = probe_columns(HOUSING_ID)
print('HOUSING columns:', housing_cols)               # <- everyone: read this once
HOUSING_DATE_COL = find_col(housing_cols, ['filing_date', 'filed_date'], regex=r'filing.*date|date.*filing')
print('Housing date column ->', HOUSING_DATE_COL)

df_housing_raw   = load_since(HOUSING_ID, HOUSING_DATE_COL)
df_transport_raw = load_since(TRANSPORT_ID, 'crash_date')
print(df_housing_raw.shape, df_transport_raw.shape)

## 1. Dataset overview and selection reasoning (M3 writes; M1 and M2 add details for their dataset)

| | Housing | Transportation |
|---|---|---|
| Dataset | DOB NOW: Build - Job Application Filings (`w9ak-ipjd`, catalog ID **H5**) | Motor Vehicle Collisions - Crashes (`h9gi-nx95`, catalog ID **T4**) |
| Size (NYC Open Data page, Oct 2026) | ~969K rows, 95 columns | ~2.27M rows, 29 columns |
| Geographic attributes | borough, street name, block, lot, BIN, community board (ZIP / lat-long: **confirm in `housing_cols`**) | borough, zip_code, latitude, longitude, on/off/cross street |
| Temporal attributes | filing date (**confirm name in `housing_cols`**) | crash_date, crash_time |
| Both have >= 9 columns | yes | yes |

**Why this pair:** both are tied to the same places and months, so they can be compared at ZIP x month without matching individual rows. Question: *do areas with heavy construction activity also see more street-safety problems?*

**Sampling (required justification):** both tables are large, so we download rows from **2022-01-01** onward through the SoQL API in 100k-row pages. 2022 onward gives ~4 years (enough for seasonality and lags), keeps the data recent, and fits in Colab memory.

**Known issues to document**
- Crash table: ZIP and borough are often blank (recovered from coordinates where possible, see section 5), coordinates of 0,0 exist, and the table is preliminary data.
- **The crash feed was reported paused** (Streetsblog, 9 Sep 2026: data stopped around 11 Jun 2026). Our pipeline therefore uses the **common date window** of both datasets and drops the last incomplete month. Print `df_transport_raw['crash_date'].max()` and write the real end date here.
- Housing table: column names differ slightly from the CSV download vs. the API (e.g. the community board column is spelled `commmunity_board`), so we look columns up with `find_col` instead of hard-coding them.

**Integration plan:** aggregate both to **ZIP x year-month**, then join on `zip` + `year_month` (details and alternatives in section 5).

## 2. Shared EDA helpers (M1 + M2)

In [ ]:
def overview(df, name):
    print(f'=== {name}: {df.shape[0]:,} rows x {df.shape[1]} cols ===')
    display(pd.DataFrame({'dtype': df.dtypes, 'nulls': df.isna().sum(),
                          'null_%': (df.isna().mean()*100).round(1), 'unique': df.nunique()}))

def iqr_bounds(s, k=1.5):
    q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    return q1 - k*i, q3 + k*i

### 2b. Shared cleaning helpers (M3 wrote these; **M1 and M2 should reuse them**)
`clean_zip`, `clean_borough`, `clean_coords`, `add_time_cols`, `fill_zip_nearest`. Using the same functions on both datasets guarantees the join keys match exactly (5-digit string ZIPs, upper-case borough names, NYC-box coordinates).

In [ ]:
from sklearn.neighbors import BallTree

BOROUGHS = ['BRONX', 'BROOKLYN', 'MANHATTAN', 'QUEENS', 'STATEN ISLAND']
BORO_MAP = {'1': 'MANHATTAN', '2': 'BRONX', '3': 'BROOKLYN', '4': 'QUEENS', '5': 'STATEN ISLAND',
            'MN': 'MANHATTAN', 'BX': 'BRONX', 'BK': 'BROOKLYN', 'QN': 'QUEENS', 'SI': 'STATEN ISLAND'}
NYC_LAT, NYC_LON = (40.45, 40.95), (-74.30, -73.65)      # rough NYC bounding box (domain rule)

def clean_zip(s):
    """Keep the first 5 digits ('11201-1234' -> '11201'); anything outside NYC ZIP range -> NaN."""
    z = s.astype(str).str.extract(r'(\d{5})')[0]
    return z.where(pd.to_numeric(z, errors='coerce').between(10001, 11697))

def clean_borough(s):
    """Upper-case, strip, map numeric codes (1-5) and abbreviations; unknown -> NaN."""
    b = s.astype(str).str.strip().str.upper().str.replace(r'\.0$', '', regex=True).replace(BORO_MAP)
    return b.where(b.isin(BOROUGHS))

def clean_coords(lat, lon):
    """Coordinates outside the NYC box (including the 0,0 placeholder) are invalid -> NaN."""
    lat, lon = pd.to_numeric(lat, errors='coerce'), pd.to_numeric(lon, errors='coerce')
    ok = lat.between(*NYC_LAT) & lon.between(*NYC_LON)
    return lat.where(ok), lon.where(ok)

def add_time_cols(df, date_col):
    out = df.copy()
    out['date'] = pd.to_datetime(out[date_col], errors='coerce')
    out['year'], out['month'] = out['date'].dt.year, out['date'].dt.month
    out['year_month'] = out['date'].dt.strftime('%Y-%m')
    return out

def fill_zip_nearest(target, ref, max_km=0.5):
    """Recover a missing ZIP from the NEAREST reference point that has a ZIP (haversine BallTree).
    Only used if the nearest point is within max_km; otherwise the ZIP stays missing."""
    out = target.copy()
    out['zip'] = out['zip'].astype(object)                  # may be all-NaN float if the table had no ZIP column
    need = out['zip'].isna() & out['lat'].notna() & out['lon'].notna()
    r = ref.dropna(subset=['zip', 'lat', 'lon'])
    if need.sum() == 0 or len(r) == 0:
        return out
    tree = BallTree(np.radians(r[['lat', 'lon']].to_numpy()), metric='haversine')
    dist, idx = tree.query(np.radians(out.loc[need, ['lat', 'lon']].to_numpy()), k=1)
    km = dist[:, 0] * 6371.0088
    out.loc[need, 'zip'] = np.where(km <= max_km, r['zip'].to_numpy()[idx[:, 0]], None)
    return out

### Cleaning contract (read before cleaning)
Your cleaned table must end up in **`df_housing`** (M1) / **`df_transport`** (M2) with exactly these columns, so integration never breaks:

| Both | Housing only (M1) | Transport only (M2) |
|---|---|---|
| `date` (datetime), `year`, `month`, `year_month` ('2024-03'), `zip` (5-char string), `borough` (upper-case), `lat`, `lon` | `job_type` | `number_of_persons_injured`, `number_of_persons_killed`, `number_of_pedestrians_injured/killed`, `number_of_cyclist_injured/killed`, `number_of_motorist_injured/killed` (numeric, no nulls) |

`quick_clean_housing` / `quick_clean_transport` in section 5 show the minimum version. Do the real cleaning (with EDA, outlier handling and markdown justifications) and **overwrite** `df_housing` / `df_transport`. Do not change the column names.

## 3. HOUSING: EDA and pre-integration cleaning (**M1**)

In [ ]:
overview(df_housing_raw, 'Housing raw')
df_housing_raw.describe(include='all').T.head(30)

In [ ]:
# TODO (M1): EDA plots (job type counts, filings by borough, filings over time, missing-value chart)
# TODO (M1): cleaning, each step with a markdown cell explaining the decision:
#   - parse filing_date, create year / month / year_month
#   - standardise borough (upper-case, strip) and ZIP (5-digit string, drop invalid)
#   - drop exact duplicates (e.g. job_filing_number)
#   - missing values: drop vs impute (justify)
#   - outliers: IQR / domain rules (justify)
df_housing = df_housing_raw.copy()

## 4. TRANSPORT: EDA and pre-integration cleaning (**M2**)

In [ ]:
overview(df_transport_raw, 'Transport raw')
df_transport_raw.describe(include='all').T.head(30)

In [ ]:
# TODO (M2): EDA plots (crashes by borough/hour/weekday, injuries distribution, missing-value chart)
# TODO (M2): cleaning, each step with a markdown cell:
#   - parse crash_date, create year / month / year_month
#   - standardise borough and ZIP; many crashes have NO borough/ZIP -> decide: drop, or recover via lat/long (justify)
#   - lat/long = 0 or outside NYC bounding box -> invalid, treat as missing
#   - duplicates (collision_id), outliers in injury counts
df_transport = df_transport_raw.copy()

## 5. INTEGRATION (**M3**)
### 5a. Quick-clean stand-ins
These give a first working `merged_data.csv` today, so the website can be built on real data while M1 and M2 do their full cleaning.

In [ ]:
def quick_clean_transport(raw):
    """Minimal standardisation so integration can run NOW. M2 replaces this with the full cleaning
    (EDA, outliers, justified decisions) as long as the OUTPUT COLUMNS stay the same."""
    d = raw.copy()
    n0 = len(d)
    if 'collision_id' in d:
        d = d.drop_duplicates('collision_id')
    d = add_time_cols(d, 'crash_date')
    out = pd.DataFrame(index=d.index)
    out['date'], out['year'], out['month'], out['year_month'] = d['date'], d['year'], d['month'], d['year_month']
    out['zip'] = clean_zip(d['zip_code'])
    out['borough'] = clean_borough(d['borough'])
    out['lat'], out['lon'] = clean_coords(d['latitude'], d['longitude'])
    for c in ['number_of_persons_injured', 'number_of_persons_killed',
              'number_of_pedestrians_injured', 'number_of_pedestrians_killed',
              'number_of_cyclist_injured', 'number_of_cyclist_killed',
              'number_of_motorist_injured', 'number_of_motorist_killed']:
        out[c] = pd.to_numeric(d[c], errors='coerce').fillna(0) if c in d else 0
    missing_before = out['zip'].isna().sum()
    out = fill_zip_nearest(out, out)                       # recover ZIP from the nearest crash that has one
    print(f'[transport] rows {n0:,} -> {len(d):,} after dedupe | ZIP missing {missing_before:,} -> {out["zip"].isna().sum():,} after coordinate recovery')
    out = out.dropna(subset=['date', 'zip'])
    print(f'[transport] kept {len(out):,} rows with valid date + ZIP')
    return out

def quick_clean_housing(raw, zip_reference=None):
    """Minimal standardisation. M1 replaces it with the full cleaning (same output columns).
    zip_reference: cleaned transport table, used to recover ZIPs if the housing table lacks them."""
    cols = raw.columns
    date_col = find_col(cols, ['filing_date', 'filed_date'], regex=r'filing.*date|date.*filing')
    type_col = find_col(cols, ['job_type'], regex=r'job.*type')
    zip_col  = find_col(cols, ['zip_code', 'zip', 'zipcode'], regex=r'^zip', required=False)
    lat_col  = find_col(cols, ['latitude', 'gis_latitude'], regex=r'^(gis_)?lat', required=False)
    lon_col  = find_col(cols, ['longitude', 'gis_longitude'], regex=r'^(gis_)?lon', required=False)
    id_col   = find_col(cols, ['job_filing_number'], required=False)
    d = raw.copy()
    n0 = len(d)
    if id_col:
        d = d.drop_duplicates(id_col)
    d = add_time_cols(d, date_col)
    out = pd.DataFrame(index=d.index)
    out['date'], out['year'], out['month'], out['year_month'] = d['date'], d['year'], d['month'], d['year_month']
    out['job_type'] = d[type_col]
    out['zip'] = clean_zip(d[zip_col]) if zip_col else np.nan
    out['borough'] = clean_borough(d['borough']) if 'borough' in d else np.nan
    if lat_col and lon_col:
        out['lat'], out['lon'] = clean_coords(d[lat_col], d[lon_col])
    else:
        out['lat'], out['lon'] = np.nan, np.nan
    missing_before = out['zip'].isna().sum()
    if zip_reference is not None:
        out = fill_zip_nearest(out, zip_reference)
    print(f'[housing] rows {n0:,} -> {len(d):,} after dedupe | ZIP column found: {bool(zip_col)} | '
          f'ZIP missing {missing_before:,} -> {out["zip"].isna().sum():,} after recovery')
    out = out.dropna(subset=['date', 'zip'])
    print(f'[housing] kept {len(out):,} rows with valid date + ZIP')
    return out

In [ ]:
df_transport_q = quick_clean_transport(df_transport_raw)
df_housing_q   = quick_clean_housing(df_housing_raw, zip_reference=df_transport_q)

# Flip to True once M1 and M2 have delivered their full cleaning (same columns, see the contract above)
USE_FULL_CLEANING = False
housing_for_join   = df_housing   if USE_FULL_CLEANING else df_housing_q
transport_for_join = df_transport if USE_FULL_CLEANING else df_transport_q

### 5b. Aggregate and join at ZIP x month

In [ ]:
SCHEMA_COLS = ['zip', 'borough', 'year', 'month', 'year_month', 'lat', 'lon',
               'filings_new_building', 'filings_alteration', 'filings_other',
               'crashes_total', 'crashes_pedestrian', 'crashes_cyclist', 'crashes_motorist',
               'injuries', 'fatalities']

def housing_group(job_type):
    t = job_type.astype(str).str.strip().str.lower()
    is_nb  = t.str.contains('new building') | t.eq('nb')
    is_alt = t.str.contains('alteration') | t.str.fullmatch(r'a[123]')
    return np.select([is_nb, is_alt], ['new_building', 'alteration'], default='other')

def aggregate_housing(h):
    """One row per ZIP x month: number of job filings by group."""
    h = h.assign(job_group=housing_group(h['job_type']))
    g = h.groupby(['zip', 'year_month', 'job_group']).size().unstack('job_group', fill_value=0)
    for c in ['new_building', 'alteration', 'other']:
        if c not in g:
            g[c] = 0
    return g[['new_building', 'alteration', 'other']].add_prefix('filings_').reset_index()

def aggregate_transport(t):
    """One row per ZIP x month: crashes (all / with pedestrian / cyclist / motorist casualties), injuries, deaths."""
    t = t.assign(
        ped=(t['number_of_pedestrians_injured'] + t['number_of_pedestrians_killed']) > 0,
        cyc=(t['number_of_cyclist_injured'] + t['number_of_cyclist_killed']) > 0,
        mot=(t['number_of_motorist_injured'] + t['number_of_motorist_killed']) > 0)
    return (t.groupby(['zip', 'year_month'])
              .agg(crashes_total=('ped', 'size'), crashes_pedestrian=('ped', 'sum'),
                   crashes_cyclist=('cyc', 'sum'), crashes_motorist=('mot', 'sum'),
                   injuries=('number_of_persons_injured', 'sum'),
                   fatalities=('number_of_persons_killed', 'sum')).reset_index())

def build_zip_month_table(housing, transport):
    """Integrate at ZIP x month. Decisions (justify in the markdown cell below):
       1. keep only ZIPs present in BOTH datasets (a ZIP seen on one side only gives no comparison)
       2. use the common date window and drop the last month if it is incomplete
       3. build a COMPLETE ZIP x month grid so 'no filings / no crashes' is an explicit 0, not a missing row"""
    zips = sorted(set(housing['zip']) & set(transport['zip']))
    h, t = housing[housing['zip'].isin(zips)], transport[transport['zip'].isin(zips)]
    start = max(h['date'].min(), t['date'].min()).to_period('M')
    end_date = min(h['date'].max(), t['date'].max())
    end = end_date.to_period('M')
    if end_date.normalize() < end.end_time.normalize():     # last month not finished -> drop it
        end = end - 1
    months = [str(p) for p in pd.period_range(start, end, freq='M')]
    h, t = h[h['year_month'].isin(months)], t[t['year_month'].isin(months)]

    grid = pd.MultiIndex.from_product([zips, months], names=['zip', 'year_month']).to_frame(index=False)
    merged = (grid.merge(aggregate_housing(h), on=['zip', 'year_month'], how='left')
                  .merge(aggregate_transport(t), on=['zip', 'year_month'], how='left'))

    # ZIP-level attributes: borough = most common borough label, lat/lon = centroid of all point records
    pts = pd.concat([h[['zip', 'borough', 'lat', 'lon']], t[['zip', 'borough', 'lat', 'lon']]])
    zip_info = pts.groupby('zip').agg(
        borough=('borough', lambda s: s.mode().iloc[0] if s.notna().any() else np.nan),
        lat=('lat', 'mean'), lon=('lon', 'mean')).reset_index()
    merged = merged.merge(zip_info, on='zip', how='left')
    merged['year'] = merged['year_month'].str[:4].astype(int)
    merged['month'] = merged['year_month'].str[5:7].astype(int)

    info = dict(zips_housing=housing['zip'].nunique(), zips_transport=transport['zip'].nunique(),
                zips_kept=len(zips), window=f'{months[0]} to {months[-1]}', months=len(months),
                grid_rows=len(grid), housing_rows_used=len(h), transport_rows_used=len(t))
    return merged, info

In [ ]:
merged, info = build_zip_month_table(housing_for_join, transport_for_join)
print(info)
merged.head()

**Justification (M3: replace the numbers with `info` from your run)**
- **Join key `zip` + `year_month`.** Individual filings and crashes do not correspond to each other, so we compare *activity levels* per neighbourhood-month. ZIP is available on both sides (or recoverable from coordinates) and gives ~180+ areas, far more than the 5 boroughs; month keeps seasonality and allows lag analysis.
- **Alternatives considered:** borough x month (only 5 units, hides local patterns); row-level spatial join (a filing has no matching crash row, so it would be meaningless); census tract (too sparse per month).
- **Only ZIPs present in both datasets are kept** (a ZIP seen on one side gives no comparison).
- **Common window, partial last month dropped** (the crash feed ends earlier than the housing feed).
- **Complete ZIP x month grid:** months with no filings or no crashes are real zeros, so they are kept as explicit 0 rows instead of missing.
- **ZIP recovery:** missing crash ZIPs are filled from the nearest crash that has one (<= 0.5 km). If the housing table has no ZIP, it is filled from the nearest crash point: this borrows the crash table's ZIP boundaries, which we list as a limitation.

## 6. POST-INTEGRATION CLEANING (**M3**) (required)

In [ ]:
COUNT_COLS = ['filings_new_building', 'filings_alteration', 'filings_other',
              'crashes_total', 'crashes_pedestrian', 'crashes_cyclist', 'crashes_motorist',
              'injuries', 'fatalities']

def post_integration_clean(merged):
    """1) NaN from the left joins = no activity -> 0   2) fix dtypes   3) drop rows with no borough/coords
       4) remove duplicates   5) enforce the agreed schema/column order"""
    m = merged.copy()
    n0 = len(m)
    print('NaNs per column BEFORE:\n', m.isna().sum()[lambda s: s > 0].to_string())
    m[COUNT_COLS] = m[COUNT_COLS].fillna(0).astype(int)
    m['zip'] = m['zip'].astype(str).str.zfill(5)
    m = m.dropna(subset=['borough', 'lat', 'lon']).drop_duplicates(['zip', 'year_month'])
    print(f'rows {n0:,} -> {len(m):,} (dropped rows without borough/coordinates, and duplicates)')
    return m[SCHEMA_COLS].sort_values(['zip', 'year_month']).reset_index(drop=True)

def validate(m):
    """Fail loudly if the table breaks the contract that the website and teammates rely on."""
    assert list(m.columns) == SCHEMA_COLS, 'columns differ from the schema contract'
    assert m.isna().sum().sum() == 0, 'nulls remain'
    assert not m.duplicated(['zip', 'year_month']).any(), 'duplicate ZIP-months'
    assert (m[COUNT_COLS] >= 0).all().all(), 'negative counts'
    assert (m['crashes_pedestrian'] <= m['crashes_total']).all(), 'pedestrian crashes exceed total'
    assert set(m['borough']) <= set(BOROUGHS), 'unexpected borough labels'
    print('VALID:', m.shape, '|', m['zip'].nunique(), 'ZIPs x', m['year_month'].nunique(), 'months')

def outlier_report(m, cols=('filings_new_building', 'crashes_total')):
    """IQR rule per ZIP total. Outliers are REPORTED, not removed: they are real high-activity ZIPs."""
    tot = m.groupby('zip')[list(cols)].sum()
    for c in cols:
        q1, q3 = tot[c].quantile([.25, .75]); hi = q3 + 1.5 * (q3 - q1)
        print(f'{c}: IQR upper bound {hi:,.0f} -> {(tot[c] > hi).sum()} ZIPs above it')

In [ ]:
final = post_integration_clean(merged)     # new NaNs from the joins -> 0, dtypes, redundant rows, schema order
validate(final)                            # fails loudly if the contract is broken
outlier_report(final)
final.describe().T

In [ ]:
# Sanity plots: do the integrated numbers look believable?
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
final.groupby('year_month')[['filings_new_building', 'crashes_total']].sum().plot(ax=ax[0], secondary_y='crashes_total', title='Monthly totals')
z = final.groupby('zip')[['filings_new_building', 'crashes_total']].sum()
sns.scatterplot(data=z, x='filings_new_building', y='crashes_total', ax=ax[1]); ax[1].set_title('ZIP totals')
sns.heatmap(final.pivot_table(index='borough', columns='month', values='crashes_total', aggfunc='sum'), ax=ax[2], cmap='YlOrRd')
ax[2].set_title('Crashes: borough x month')
plt.tight_layout(); plt.show()

In [ ]:
# Export: THIS file is what the website reads (put it in the repo at data/merged_data.csv)
final.to_csv('merged_data.csv', index=False)
try:
    from google.colab import files
    files.download('merged_data.csv')
except ImportError:
    pass
print(final.shape)

## 7. Research questions and visualizations (2 per member)
Write each question + 1 chart + 2-3 sentences of interpretation (no causal claims). The same function goes into your `charts_mX.py` file in the repo.

### M1: RQ1 and RQ2
*Question 1:* TODO

*Question 2:* TODO

In [ ]:
# TODO (M1): chart for RQ1

# TODO (M1): chart for RQ2

### M2: RQ3 and RQ4
*Question 1:* TODO

*Question 2:* TODO

In [ ]:
# TODO (M2): chart for RQ3

# TODO (M2): chart for RQ4

### M3: RQ5 and RQ6
*Question 1:* TODO

*Question 2:* TODO

In [ ]:
# TODO (M3): chart for RQ5

# TODO (M3): chart for RQ6

### M4: RQ7 and RQ8
*Question 1:* TODO

*Question 2:* TODO

In [ ]:
# TODO (M4): chart for RQ7

# TODO (M4): chart for RQ8

## 8. Story: question, discovery, limits, impact (M4 with everyone's input)
TODO

## 9. Prompt log (everyone)
| Member | Step | Prompt used |
|---|---|---|
| | | |

## 10. Contributions
TODO: per member: RQs, charts, cleaning, integration, website parts.